## UAV Intrusion Detection Stacked Ensemble
This notebook implements a meta-classifier ensemble using One-Class SVM, Autoencoder, and Local Outlier Factor as described in the Whelan et al. (2020) paper into the UAV Attack Dataset by Jason Whelan, et al.

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report

### Load the Dataset

In [ ]:
FILE_PATH: str = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
df = pd.read_csv(FILE_PATH)
df.head()

,timestamp,q[0],q[1],q[2],q[3],delta_q_reset[0],delta_q_reset[1],delta_q_reset[2],delta_q_reset[3],quat_reset_counter,...,v_z_valid,xy_reset_counter,z_reset_counter,vxy_reset_counter,vz_reset_counter,heading_reset_counter,xy_global,z_global,dist_bottom_valid,label
0,258725029,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,benign
1,258842609,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,benign
2,258847608,-0.325207,0.010554,-0.006518,0.945561,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,benign
3,258852607,-0.325204,0.010556,-0.006517,0.945562,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,benign
4,258902598,-0.325214,0.010552,-0.006501,0.945559,0.999995,-4.610000e-10,1.580000e-09,0.003237,3,...,1,2,0,2,1,3,1,1,1.0,benign


### Preprocessing

In [14]:
# Prepare features and labels
X = df.drop(columns=["label", "timestamp"])
y = df["label"].map({"benign": 0, "malicious": 1})  # encode labels

# Use benign data to train unsupervised models
X_train_raw = X[y == 0]
X_test_raw = X
y_test = y

# Normalize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85)
X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95)
X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

### Train Base Models

In [15]:
# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_train_pca_svm_lof)
ocsvm_pred = np.where(ocsvm.predict(X_test_pca_svm_lof) == -1, 1, 0)

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_train_pca_svm_lof)
lof_pred = np.where(lof.predict(X_test_pca_svm_lof) == -1, 1, 0)

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=42)
ae.fit(X_train_pca_ae, X_train_pca_ae)

# Compute reconstruction error
recon = ae.predict(X_test_pca_ae)
recon_error = np.mean((X_test_pca_ae - recon) ** 2, axis=1)

# Threshold based on 97.2% percentile from training reconstruction error
train_recon = ae.predict(X_train_pca_ae)
train_error = np.mean((X_train_pca_ae - train_recon) ** 2, axis=1)
threshold = np.percentile(train_error, 97.2)
ae_pred = np.where(recon_error > threshold, 1, 0)

/home/ameens/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


### Meta-Classifier Ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np

# Stack predictions from the 3 models
stacked_features = np.vstack([ocsvm_pred, lof_pred, ae_pred]).T

# Meta-classifier (Random Forest)
meta_clf = RandomForestClassifier(random_state=42)
meta_clf.fit(stacked_features, y_test)
stacked_pred = meta_clf.predict(stacked_features)

# Evaluation
print(classification_report(
    y_test, 
    stacked_pred, 
    labels=[0, 1],  # 0 = benign, 1 = malicious
    target_names=["benign", "malicious"],
    zero_division=0  # Optional: avoid division by zero warnings
))


              precision    recall  f1-score   support

      benign       1.00      0.99      1.00      3124
   malicious       0.96      1.00      0.98       498

    accuracy                           0.99      3622
   macro avg       0.98      0.99      0.99      3622
weighted avg       0.99      0.99      0.99      3622

